# Phase 1: Spurgeon Continued Pretraining — Step 9: Evaluation & Export (Notebook C)

This notebook loads the trained Phase 1 PEFT QLoRA adapter checkpoints, evaluates its perplexity score on the 50-sermon holdout dataset, runs qualitative style validations on test prompts, and exports the final adapter weights.

## 1. Install Dependencies

In [ ]:
# Install Unsloth and specific patched versions for Kaggle environment
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Load Model and LoRA Adapter

In [ ]:
from unsloth import FastLanguageModel
import torch

# Set the path pointing to the mounted output of Run 2 (checkpoint-432)
LORA_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-432"
MAX_SEQ_LENGTH = 2048

# Load base model + LoRA adapter in 4-bit quantization natively
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LORA_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

## 3. Perplexity Evaluation on Holdout Set

In [ ]:
import math
from datasets import load_from_disk

# Set model to inference mode
FastLanguageModel.for_inference(model)

# Load holdout dataset built in Notebook A
holdout_dataset = load_from_disk("/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_holdout_dataset")

total_loss = 0.0
total_tokens = 0

print("Evaluating holdout perplexity...")
for idx, doc in enumerate(holdout_dataset):
    # Tokenize sequence
    inputs = tokenizer(doc["text"], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    num_tokens = inputs["input_ids"].size(1)
    
    # Truncate sequence safely to fit context limits without VRAM spikes
    if num_tokens > MAX_SEQ_LENGTH:
        inputs = {k: v[:, :MAX_SEQ_LENGTH] for k, v in inputs.items()}
        num_tokens = MAX_SEQ_LENGTH
        
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()
        
    total_loss += loss * num_tokens
    total_tokens += num_tokens

average_loss = total_loss / total_tokens
perplexity = math.exp(average_loss)

print(f"\nEvaluation Results:")
print(f"  - Total Holdout Tokens: {total_tokens:,}")
print(f"  - Length-Weighted Loss: {average_loss:.4f}")
print(f"  - Holdout Perplexity: {perplexity:.2f}")

## 4. Qualitative Style Validation

In [ ]:
prompts = [
    "The love of Christ is not a cold, speculative thing. It is ",
    "Text: Romans 8:28. 'And we know that all things work together for good to them that love God.' My dear friends, ",
    "What, then, is saving faith? Let us examine this question carefully, for "
]

print("Running qualitative generations...\n")
for i, prompt in enumerate(prompts):
    print(f"--- Prompt {i+1} ---")
    print(f"Input: {prompt}")
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=0.7, 
            top_p=0.9, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output:\n{generated_text}\n")

## 5. Save Final PEFT Adapter

In [ ]:
output_path = "/kaggle/working/spurgeon_lora_final"
print(f"Saving final LoRA adapter to {output_path}...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print("Weights saved successfully. Phase 1 pretraining evaluation and export completed.")

## 6. Convert and Export to GGUF (Original 16-bit)

We merge our PEFT adapter weights back into the base Qwen model and export directly to GGUF format. We specify `quantization_method = "f16"` to preserve the original 16-bit floating point precision of the weights without any quantization loss.

In [ ]:
# Convert to GGUF format locally on Kaggle. 
# Unsloth will compile/download llama.cpp internally and output the merged F16 GGUF file.
model.save_pretrained_gguf(
    "/kaggle/working/spurgeon_f16_gguf",
    tokenizer,
    quantization_method = "f16",
)

## 7. Upload GGUF Model to Hugging Face Hub

We securely load your Hugging Face write token from Kaggle Secrets, initialize the Hugging Face API, and upload the 6GB `unsloth.F16.gguf` file directly to your model repository.

In [ ]:
# Upload GGUF file to Hugging Face Hub using Kaggle Secrets
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
import glob
import os

try:
    # Load Hugging Face API write token from Kaggle Secrets (Add 'HF_TOKEN' in Add-ons -> Secrets)
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    
    api = HfApi()
    repo_id = "rafaelvieirar1r/qwen2.5-3b-spurgeon-gguf-phase1"
    
    print(f"Creating Hugging Face repository {repo_id} (if it does not exist)...")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    
    # Dynamically find any .gguf file under /kaggle/working/
    gguf_files = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
    if not gguf_files:
        raise FileNotFoundError("No GGUF file found under /kaggle/working")
    
    file_path = gguf_files[0]
    file_name = os.path.basename(file_path)
    #file_path = "/kaggle/working/spurgeon_f16_gguf_gguf/qwen2.5-3b.F16.gguf"

    print(f"Found GGUF file: {file_path}")
    print(f"Uploading {file_name} to Hugging Face Hub...")
    api.upload_file(
        path_or_fileobj=file_path,
        path_in_repo=file_name,
        repo_id=repo_id,
        repo_type="model"
    )
    print("Model successfully uploaded to Hugging Face!")
except Exception as e:
    print(f"Failed to upload to Hugging Face: {e}")
    print("Please ensure you have toggled Internet ON and added the 'HF_TOKEN' key to Kaggle Secrets.")